### 🛠️ Cell 1: Environment Setup & Hardware Acceleration
**Function:** Imports necessary libraries, fixes potential system duplicate library crashes, and initializes the NVIDIA GPU (CUDA) if available to speed up processing.

In [ ]:
# =====================================================================
# 🛠️ CELL 1 — CORE ENVIRONMENT SETUP & HARDWARE COMPUTE
# =====================================================================
import os
import time
import cv2
import torch
import numpy as np
from ultralytics import YOLO

# 1. Initialize environment and resolve system library duplicate issues
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 2. Enable GPU hardware acceleration if available
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    print(f"[INFO] Hardware acceleration enabled on: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] CUDA GPU not detected. Using CPU.")

### 🧠 Cell 2: Deep Learning Model Initialization
**Function:** Loads the YOLOv8 weights into the active memory. Keeping this in a separate cell prevents reloading the heavy model every time you want to test the video.

In [ ]:
# =====================================================================
# 🧠 CELL 2 — MODEL ARCHITECTURE LOADING
# =====================================================================
# 3. Load YOLO model weights for object detection and tracking
model = YOLO("yolo26l.pt", task="detect")
print("[SUCCESS] YOLO Model loaded successfully.")

### ⚙️ Cell 3: Configuration & File Paths
**Function:** Initializes global tracking variables, counters, and sets up the input/output file paths. **(Update your video path here)**.

In [ ]:
# =====================================================================
# ⚙️ CELL 3 — CONFIGURATION & METRICS INITIALIZATION
# =====================================================================
# Global counters and state dictionaries
total_in = 0
total_out = 0
track_history = {}
crossed_direction = {}
entry_timestamps = {}
dwell_durations = {}

# =====================================================================
# 🔴 🚨 URGENT: CHANGE THE VIDEO PATH HERE 🚨 🔴
# =====================================================================
# Replace "People_Walking.mp4" with the exact path to your local video.
video_file_path = "People_Walking.mp4"  # <--- INSERT YOUR VIDEO PATH HERE

output_file_path = "output_analytics.mp4"
window_name = "OFFLINE VIDEO ANALYSIS"

print(f"[INFO] Target video file configured: {video_file_path}")

### 🎬 Cell 4: Main Video Processing & Analytics Pipeline
**Function:** Opens the video file, runs frame-by-frame YOLO tracking, calculates crossing logic and dwell times, renders the UI overlays, and exports the final MP4 video.

In [ ]:
# =====================================================================
# 🎬 CELL 4 — CORE INFERENCE & AUTOSAVE PIPELINE
# =====================================================================
print(f"[INFO] Opening video file...")

# 4. Open the video file using OpenCV (without FFMPEG to avoid bugs)
cap = cv2.VideoCapture(video_file_path)

if not cap.isOpened():
    print(f"[ERROR] Cannot open '{video_file_path}'. Please check the path and file format!")
else:
    # Extract original video properties to match output quality
    video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30
    
    # Initialize video codec for saving the output file (MP4V format)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_file_path, fourcc, video_fps, (video_width, video_height))
    
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    # 5. Main Processing Loop
    while True:
        start_time = time.time()
        ret, frame = cap.read()
        
        # Break the loop gracefully if the video ends
        if not ret:
            print("[INFO] Video file reached the end. Finalizing and saving...")
            break

        h, w, _ = frame.shape
        line_x_pixel = int(w * 0.50)  # 50% midpoint line for crossing threshold
        
        # Draw the physical tracking line on the screen
        cv2.line(frame, (line_x_pixel, 0), (line_x_pixel, h), (0, 0, 255), 3)

        # Execute object tracking using the YOLO model (ByteTrack)
        results = model.track(frame, imgsz=640, conf=0.25, tracker="bytetrack.yaml", persist=True, verbose=False)

        # 🟢 Define empty list to avoid NameError if the current frame has no detections
        ids = [] 
        
        if results[0].boxes.id is not None:
            # Extract tracking variables and move them to CPU/Numpy for processing
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            clses = results[0].boxes.cls.cpu().numpy().astype(int)

            current_epoch = time.time()

            for box, track_id, cls_id in zip(boxes, ids, clses):
                # Calculate the center X coordinate of the bounding box
                center_x = int((box[0] + box[2]) / 2)

                # Determine state based on the person's position relative to the line
                current_state = "IN" if center_x > line_x_pixel else "OUT"
                previous_state = crossed_direction.get(track_id, None)

                # Increment global counters if the crossing line is breached
                if previous_state is not None and previous_state != current_state:
                    if current_state == "IN": 
                        total_in += 1
                    elif current_state == "OUT": 
                        total_out += 1

                # Record exact entry timestamp for a newly established state
                if previous_state != current_state:
                    entry_timestamps[track_id] = current_epoch

                # Update states and historical coordinates
                crossed_direction[track_id] = current_state
                track_history[track_id] = center_x
                
                # Calculate Dwell Time (Total seconds spent in current state)
                elapsed_seconds = current_epoch - entry_timestamps[track_id]
                dwell_durations[track_id] = elapsed_seconds

                # Define bounding box colors and text labels based on state/class
                if cls_id == 1:
                    bounding_color, label = (255, 0, 255), "EMP"  # Purple for Employee
                elif current_state == "IN":
                    bounding_color, label = (0, 255, 0), "IN"     # Green for Inside
                else:
                    bounding_color, label = (0, 0, 255), "OUT"    # Red for Outside

                # Render bounding boxes and info text overlays on the frame
                cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), bounding_color, 2)
                cv2.putText(frame, f"{label} #{track_id} | {int(elapsed_seconds)}s", (int(box[0]), int(box[1]) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, bounding_color, 2)

        # Periodically clean up memory dictionary to prevent RAM bloat
        if len(track_history) > 300:
            for key in list(track_history.keys())[:-100]:
                track_history.pop(key, None)
                crossed_direction.pop(key, None)
                entry_timestamps.pop(key, None)
                dwell_durations.pop(key, None)

        # Calculate high-level statistics for the dashboard
        net_inside = max(0, total_in - total_out)
        fps = 1.0 / max(time.time() - start_time, 0.001)

        inside_times = [dwell_durations[tid] for tid in dwell_durations if crossed_direction.get(tid) == "IN" and (tid in ids)]
        avg_time = sum(inside_times) / len(inside_times) if inside_times else 0.0

        # Construct and render the UI top overlay dashboard
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 55), (20, 20, 20), -1)
        cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)
        
        header = f"IN: {total_in} | OUT: {total_out} | INSIDE: {net_inside} | AVG TIME: {int(avg_time)}s | FPS: {round(fps, 1)}"
        cv2.putText(frame, header, (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

        # Write frame to output video file and display it
        video_writer.write(frame)
        cv2.imshow(window_name, frame)

        # Press 'q' key for early termination
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("[INFO] 'q' pressed. Stopping early...")
            break

    # 6. Close and release system hardware resources safely
    cap.release()
    video_writer.release()
    cv2.destroyAllWindows()
    print(f"[SUCCESS] Video successfully compiled and saved as: '{output_file_path}'")